# Recurring Charge ML Experimentation

This notebook connects to the local PostgreSQL database populated from DynamoDB and implements the industry-standard multi-layered enrichment pipeline.

In [2]:
import psycopg2
import pandas as pd
import numpy as np
import re
from sentence_transformers import SentenceTransformer
from sklearn.cluster import DBSCAN
import warnings
warnings.filterwarnings('ignore')

# Connect to PostgreSQL
conn = psycopg2.connect(
    host="db",
    port="5432",
    user="ml_user",
    password="ml_password",
    database="recurring_charges_ml"
)

# Load transactions
query = "SELECT * FROM transactions"
df_tx = pd.read_sql(query, conn)
print(f"Loaded {len(df_tx)} transactions.")

Loaded 30 transactions.


## 1. Normalization & Cleaning Pipeline

Strip noise (IDs, dates), standardize entities (LTD, INC), and extract geography.

In [3]:
def clean_merchant_name(name):
    if not isinstance(name, str) or not name: return ""
    name = name.upper()

    # Replace HTML ampersand entities and literal ampersands with " AND "
    name = re.sub(r'&(?:AMP;)?', ' AND ', name)
    
    # Remove Transaction IDs (*1234, #A1B2)
    name = re.sub(r'[*#]\s?(?=[A-Za-z]*\d)[A-Z0-9]+', ' ', name)
    
    # Remove Dates (24/02, etc)
    name = re.sub(r'\d{2}/\d{2}', ' ', name)
    
    # Remove Payment Methods & Region Codes (APPLEPAY, GB)
    name = re.sub(r'\b(?:APPLEPAY|GB|UK)\b', ' ', name)

    # 1. Remove Zettle and all its attached garbage (ZETTLE_, ZETTLE_*, IZETTLE)
    name = re.sub(r'\bI?ZETTLE[_*]*', ' ', name)
    
    # 2. Remove PayPal and its attached garbage (PAYPAL *, PAYPAL_)
    name = re.sub(r'\bPAYPAL\s*[*_]*', ' ', name)
    
    # 3. Standard Payment Methods & Region Codes
    name = re.sub(r'\b(?:APPLEPAY|SUMUP|SQUARE|GB|UK)\b', ' ', name)

    
    # Remove long standalone numbers (Store IDs, Card last 4, Account numbers)
    # Matches any standalone number that is 3 or more digits long (e.g., 2663, 9178, 1448432524)
    name = re.sub(r'\b\d{3,}\b', ' ', name)
    
    # Remove alphanumeric reference numbers (e.g., 08477317OB, T1024025862)
    # The (?=.*\d) ensures the 8+ character word actually contains a number
    # so we don't accidentally delete purely text names like "SAINSBURYS"
    name = re.sub(r'\b(?=[A-Za-z]*\d)[A-Za-z0-9]{8,}\b', ' ', name)
    
    # Remove reference numbers containing dashes (e.g., GB25682264-000035, V72413-23041)
    name = re.sub(r'\b[A-Z0-9]*\d[A-Z0-9]*-[A-Z0-9]+\b', ' ', name)

    
    # Standardize entities
    name = re.sub(r'\b(LTD|LIMITED|INC|CORP|LLC|PLC)\b', ' ', name)

    # Remove website prefixes and suffixes
    name = re.sub(r'\b(WWW\.|\\.COM|\\.CO\\.UK)\b', ' ', name)

    # Add this inside your clean_merchant_name function
    name = re.sub(r'\bPYMT\b', 'PAYMENT', name)
    name = re.sub(r'\bS/MKTS\b', 'SUPERMARKET', name)
    name = re.sub(r'\bTXN\b', 'TRANSACTION', name)
    
    # Clean up whitespace
    name = ' '.join(name.split())
    
    return name


if len(df_tx) > 0:
    df_tx['cleaned_description'] = df_tx['description'].apply(clean_merchant_name)
    display(df_tx[['description', 'cleaned_description']].head())

,description,cleaned_description
0,Interest On Your Standard Balanc,INTEREST ON YOUR STANDARD BALANC
1,"Temu.com, London","TEMU.COM, LONDON"
2,"Payment, Thank You","PAYMENT, THANK YOU"
3,Interest On Your Standard Balanc,INTEREST ON YOUR STANDARD BALANC
4,Interest On Your Standard Balanc,INTEREST ON YOUR STANDARD BALANC


In [4]:
def remove_dd_mmm_dates(name):
    if not isinstance(name, str) or not name: return ""
    
    # 1. Remove dd MMM dates (e.g., ON 03 DEC, 01 JUN)
    name = re.sub(r'\b(?:ON\s+)?\d{1,2}\s(?:JAN|FEB|MAR|APR|MAY|JUN|JUL|AUG|SEP|OCT|NOV|DEC)\b', '', name)
    
    # 2. Remove common UK bank transaction codes (usually standalone words)
    # BCC=Bank Credit Card, CLP/CPM=Contactless, DDR=Direct Debit, FT=Funds Transfer, etc.
    name = re.sub(r'\b(?:BCC|CLP|CPM|DDR|DD|FT|BC|CL|CP|DC|SST|UNP|ASD)\b', '', name)
    
    # 3. Clean up any excess whitespace left behind
    name = ' '.join(name.split())
    return name


if len(df_tx) > 0:
    df_tx['cleaned_description'] = df_tx['cleaned_description'].apply(remove_dd_mmm_dates)
    display(df_tx[['description', 'cleaned_description']].head())


,description,cleaned_description
0,Interest On Your Standard Balanc,INTEREST ON YOUR STANDARD BALANC
1,"Temu.com, London","TEMU.COM, LONDON"
2,"Payment, Thank You","PAYMENT, THANK YOU"
3,Interest On Your Standard Balanc,INTEREST ON YOUR STANDARD BALANC
4,Interest On Your Standard Balanc,INTEREST ON YOUR STANDARD BALANC


In [5]:
# Export all transactions with their matched labels to a CSV for easy bulk viewing
df_tx[['description', 'cleaned_description']].to_csv('cleaned_transactions_preview.csv', index=False)
print(f"Saved {len(df_tx)} rows to cleaned_transactions_preview.csv!")


Saved 30 rows to cleaned_transactions_preview.csv!


## 2. Feature Engineering: Semantic Embeddings

Use lightweight NLP model to catch semantic relationships instead of just TF-IDF.

In [6]:
# Load lightweight embedding model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Generate embeddings for cleaned descriptions
if len(df_tx) > 0:
    print("Generating embeddings (this may take a moment)...")
    embeddings = model.encode(df_tx['cleaned_description'].tolist())
    print(f"Generated embeddings of shape {embeddings.shape}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Generating embeddings (this may take a moment)...
Generated embeddings of shape (30, 384)


In [7]:
# Create DBSCAN instance
# eps defines the maximum distance between two samples to be grouped
# min_samples is the minimum number of transactions to form a valid merchant cluster
clustering_model = DBSCAN(eps=0.3, min_samples=3, metric='cosine')

# Fit the model to your generated embeddings
cluster_labels = clustering_model.fit_predict(embeddings)

# Assign the predicted categories back to your DataFrame
df_tx['merchant_cluster_id'] = cluster_labels

# View the results (excluding noise points labeled as -1 which didn't fit into any cluster)
clustered_tx = df_tx[df_tx['merchant_cluster_id'] != -1]

# Display the clusters so you can verify if the categoriser worked!
display(clustered_tx[['description', 'cleaned_description', 'merchant_cluster_id']].sort_values('merchant_cluster_id').head(30))


,description,cleaned_description,merchant_cluster_id
0,Interest On Your Standard Balanc,INTEREST ON YOUR STANDARD BALANC,0
3,Interest On Your Standard Balanc,INTEREST ON YOUR STANDARD BALANC,0
4,Interest On Your Standard Balanc,INTEREST ON YOUR STANDARD BALANC,0
6,Interest On Your Standard Balanc,INTEREST ON YOUR STANDARD BALANC,0
7,Interest On Your Standard Balanc,INTEREST ON YOUR STANDARD BALANC,0
16,Interest On Your Standard Balanc,INTEREST ON YOUR STANDARD BALANC,0
14,Interest On Your Standard Balanc,INTEREST ON YOUR STANDARD BALANC,0
13,Interest On Your Standard Balanc,INTEREST ON YOUR STANDARD BALANC,0
20,Interest On Your Standard Balanc,INTEREST ON YOUR STANDARD BALANC,0
24,Interest On Your Standard Balanc,INTEREST ON YOUR STANDARD BALANC,0


In [8]:
# Export all transactions with their matched labels to a CSV for easy bulk viewing
df_tx[['description', 'cleaned_description', 'merchant_cluster_id']].sort_values('merchant_cluster_id').to_csv('cleaned_transactions_preview.csv', index=False)
print(f"Saved {len(df_tx)} rows to cleaned_transactions_preview.csv!")

Saved 30 rows to cleaned_transactions_preview.csv!


## 3. Categorization / Classification

We can use the  embeddings we already generated to map either clusters or individual transactions to pre-defined spending categories.

## 3a. Categorization / Classification (V1: Taxonomy)

Uses the original accounting-style categories.

In [9]:
import re
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# --- V1: Existing Taxonomy Approach ---
category_map_v1 = {
    "Housing": "Housing rent and mortgage payments",
    "Utilities": "Household Utilities electricity water and gas",
    "Telecom": "Telecom internet broadband and mobile phone bills",
    "Subscriptions": "Recurring subscriptions software gym and streaming",
    "Insurance": "Insurance premiums auto home and health",
    "Groceries": "Groceries and supermarket shopping",
    "Dining": "Restaurants dining out and coffee shops",
    "Delivery": "Food delivery and takeaways",
    "Transport": "Public transport trains and buses",
    "Auto": "Auto expenses fuel parking and maintenance",
    "Shopping": "Retail shopping clothing electronics and home",
    "Entertainment": "Entertainment events and cinema",
    "Health": "Health pharmacy and personal care",
    "Travel": "Travel flights and hotels",
    "Bank Fees": "Bank fees and financial charges",
    "Transfers": "Money transfers and savings",
    "Income": "Income salary and refunds",
    "Household": "Repairs, DIY, decorating, pet care",
    "Unknown": "Cash widthdrawls"
}
category_labels_v1 = list(category_map_v1.keys())
category_descriptions_v1 = list(category_map_v1.values())

regex_rules_v1 = {
    r'\b(?:TESCO|SAINSBURYS|ASDA|WAITROSE|ALDI|LIDL|MIMEO)\b': 'Groceries',
    r'\b(?:HSBC|BARCLAYS|LLOYDS|NATWEST|HALIFAX|SANTANDER|MONZO|MOORCROFT GROUP)\b': 'Bank Fees',
    r'\b(?:MCDONALDS|KFC|BURGER KING|NANDOS|COSTA|STARBUCKS|UPPERCRUST|FULLER SMITH|JD WETHERSPOON)\b': 'Dining',
    r'\b(?:UBER|UBEREATS|DELIVEROO|JUST EAT)\b': 'Delivery',
    r'\b(?:TFL|TRAINLINE|RAIL|ARRIVA|STAGECOACH)\b': 'Transport',
    r'\b(?:NETFLIX|SPOTIFY|AMAZON PRIME|DISNEY PLUS|CURSOR, AI|UNISON|NOW)\b': 'Subscriptions',
    r'\b(?:MAX SPIELMANN|NYX|EDUBOX|EDE AND RAVENSCROF|WH SMITH|TGTG)|AMIGO @ HAMMERSMIT|MR SIMMS OLDE SWEE|WELCOME BREAK-LOND|DUNELM|SIMMONS|AMZNMKTPLACE\b': 'Shopping',
    r'\b(?:ANIMAL FRIENDS INS|COMPANION C|PETSATHOME|SWAROVSKI|PL2)\b': 'Household',
    r'\b(DVLA-[A-Z]{2}\d{2}\s?[A-Z]{3}|SHELL)\b': 'Auto',
    r'\b(AO-OPTICALSERVICES|ADARO. VISIONEXPRE)\b': 'Health',
    r'\b(MIMEO|SA_SUPPORT)\b': 'Subscriptions',
    r'\b(SCREWFIX)\b': 'Household',
    r'\b(EE D)\b': 'Telecom',
    r'\b(NOTEMACHINE NOTEMACHINE)\b': 'Unknown'
}

df_tx['suggested_category_v1'] = None
df_tx['category_confidence_v1'] = 0.0
df_tx['match_type_v1'] = None

unmatched_clusters_v1 = []
unmatched_noise_indices_v1 = []

for cluster_id in df_tx['merchant_cluster_id'].unique():
    bool_idx = df_tx['merchant_cluster_id'] == cluster_id
    cluster_texts = df_tx.loc[bool_idx, 'cleaned_description'].tolist()
    
    if cluster_id == -1:
        for idx in df_tx[bool_idx].index:
            text = df_tx.loc[idx, 'cleaned_description']
            matched = False
            for pattern, category in regex_rules_v1.items():
                if re.search(pattern, text):
                    df_tx.at[idx, 'suggested_category_v1'] = category
                    df_tx.at[idx, 'category_confidence_v1'] = 1.0
                    df_tx.at[idx, 'match_type_v1'] = 'Rule'
                    matched = True
                    break
            if not matched:
                unmatched_noise_indices_v1.append(idx)
        continue
        
    cluster_resolved = False
    for pattern, category in regex_rules_v1.items():
        if any(re.search(pattern, text) for text in cluster_texts):
            df_tx.loc[bool_idx, 'suggested_category_v1'] = category
            df_tx.loc[bool_idx, 'category_confidence_v1'] = 1.0
            df_tx.loc[bool_idx, 'match_type_v1'] = 'Rule'
            cluster_resolved = True
            break
            
    if not cluster_resolved:
        unmatched_clusters_v1.append(cluster_id)
        
print(f"Phase 1: Rules Engine matched {len(df_tx[df_tx['match_type_v1'] == 'Rule'])} transactions.")

category_embeddings_v1 = model.encode(category_descriptions_v1)

if unmatched_noise_indices_v1:
    noise_embeddings = embeddings[unmatched_noise_indices_v1]
    sim_matrix = cosine_similarity(noise_embeddings, category_embeddings_v1)
    best_indices = np.argmax(sim_matrix, axis=1)
    best_scores = np.round(np.max(sim_matrix, axis=1), 2)
    
    df_tx.loc[unmatched_noise_indices_v1, 'suggested_category_v1'] = [category_labels_v1[i] for i in best_indices]
    df_tx.loc[unmatched_noise_indices_v1, 'category_confidence_v1'] = best_scores
    df_tx.loc[unmatched_noise_indices_v1, 'match_type_v1'] = 'ML'

for cluster_id in unmatched_clusters_v1:
    bool_idx = df_tx['merchant_cluster_id'] == cluster_id
    cluster_embeddings = embeddings[bool_idx]
    
    centroid = np.mean(cluster_embeddings, axis=0).reshape(1, -1)
    sim_matrix = cosine_similarity(centroid, category_embeddings_v1)
    best_idx = np.argmax(sim_matrix, axis=1)[0]
    best_score = np.round(np.max(sim_matrix, axis=1)[0], 2)
    
    df_tx.loc[bool_idx, 'suggested_category_v1'] = category_labels_v1[best_idx]
    df_tx.loc[bool_idx, 'category_confidence_v1'] = best_score
    df_tx.loc[bool_idx, 'match_type_v1'] = 'ML'
    
print(f"Phase 2: Semantic ML matched {len(df_tx[df_tx['match_type_v1'] == 'ML'])} transactions.")

display_cols = ['cleaned_description', 'merchant_cluster_id', 'match_type_v1', 'suggested_category_v1', 'category_confidence_v1']
display(df_tx[display_cols].sample(10))

Phase 1: Rules Engine matched 1 transactions.
Phase 2: Semantic ML matched 29 transactions.


,cleaned_description,merchant_cluster_id,match_type_v1,suggested_category_v1,category_confidence_v1
28,"ECONOMIST THE,",-1,ML,Groceries,0.27
0,INTEREST ON YOUR STANDARD BALANC,0,ML,Unknown,0.33
7,INTEREST ON YOUR STANDARD BALANC,0,ML,Unknown,0.33
15,"THESELFSTORAGECOMPAN, HEMEL",-1,ML,Shopping,0.31
25,"PAYMENT, THANK YOU",1,ML,Unknown,0.27
17,"PAYMENT, THANK YOU",1,ML,Unknown,0.27
8,"PAYMENT, THANK YOU",1,ML,Unknown,0.27
6,INTEREST ON YOUR STANDARD BALANC,0,ML,Unknown,0.33
24,INTEREST ON YOUR STANDARD BALANC,0,ML,Unknown,0.33
23,"PAYMENT, THANK YOU",1,ML,Unknown,0.27


In [10]:
output_cols = [
    'description', 
    'cleaned_description', 
    'merchant_cluster_id', 
    'suggested_category_v1', 
    'category_confidence_v1'
]
df_tx['category_confidence_v1'] = df_tx['category_confidence_v1'].round(3)
df_tx[output_cols].sort_values(['category_confidence_v1', 'suggested_category_v1']).to_csv('cleaned_transactions_preview_v1.csv', index=False)
print(f"Saved {len(df_tx)} categorized rows to cleaned_transactions_preview_v1.csv!")

Saved 30 categorized rows to cleaned_transactions_preview_v1.csv!


## 3b. Categorization / Classification (V2: Behavioural)

Uses the new user-centric behavioral categories.

In [11]:
import re
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# --- V2: New Behavioral Approach ---
category_map_v2 = {
    "Housing & Utilities": "Essential housing mortgage rent council tax electricity gas water heating bills DIY repairs",
    "Insurance & Finance": "Essential insurance premiums auto home health life bank fees loan repayments financial charges",
    "Telecom & Software": "Fixed recurring telecom internet broadband mobile phone software productivity subscriptions",
    "Groceries": "Essential groceries supermarket food shopping household supplies toiletries",
    "Transport & Car": "Essential public transport trains buses commute car fuel petrol parking car maintenance",
    "Health & Care": "Essential health pharmacy medical dental personal care therapy childcare petcare",
    "Dining Out": "Discretionary eating out restaurants cafes coffee shops pubs bars social dining",
    "Takeaways & Delivery": "Discretionary fast food takeaways convenient food delivery apps",
    "Shopping & Retail": "Discretionary retail shopping clothing apparel electronics home upgrades hobbies gifts",
    "Entertainment & Leisure": "Discretionary entertainment events cinema gym memberships media streaming tv video games",
    "Travel & Holidays": "Discretionary travel tourism flights hotels vacations holidays leisure trips",
    "Savings & Investments": "Wealth transfers savings accounts investments stocks crypto pensions",
    "Income": "Incoming money salary wage payroll dividends refunds cashback",
    "Cash & Unknown": "Cash withdrawals ATM transfers to friends unknown expenses miscellaneous"
}
category_labels_v2 = list(category_map_v2.keys())
category_descriptions_v2 = list(category_map_v2.values())

regex_rules_v2 = {
    r'\b(?:TESCO|SAINSBURYS|ASDA|WAITROSE|ALDI|LIDL|MIMEO|B AND M)\b': 'Groceries',
    r'\b(?:HSBC|BARCLAYS|LLOYDS|NATWEST|HALIFAX|SANTANDER|MONZO|MOORCROFT GROUP)\b': 'Insurance & Finance',
    r'\b(?:MCDONALDS|KFC|BURGER KING|NANDOS|COSTA|STARBUCKS|UPPERCRUST|FULLER SMITH|JD WETHERSPOON)\b': 'Dining Out',
    r'\b(?:UBER|UBEREATS|DELIVEROO|JUST EAT)\b': 'Takeaways & Delivery',
    r'\b(?:TFL|TRAINLINE|RAIL|ARRIVA|STAGECOACH)\b': 'Transport & Auto',
    r'\b(?:NETFLIX|SPOTIFY|AMAZON PRIME|DISNEY PLUS|CURSOR, AI|UNISON|NOW)\b': 'Entertainment & Leisure',
    r'\b(?:MAX SPIELMANN|NYX|EDUBOX|EDE AND RAVENSCROF|WH SMITH|TGTG|AMIGO @ HAMMERSMIT|MR SIMMS OLDE SWEE|WELCOME BREAK-LOND|DUNELM|SIMMONS|AMZNMKTPLACE|H AND M)\b': 'Shopping & Retail',
    r'\b(?:ANIMAL FRIENDS INS|COMPANION C|PETSATHOME)\b': 'Health & Care', 
    r'\b(DVLA-[A-Z]{2}\d{2}\s?[A-Z]{3}|SHELL)\b': 'Transport & Auto',
    r'\b(AO-OPTICALSERVICES|ADARO. VISIONEXPRE)\b': 'Health & Care',
    r'\b(SA_SUPPORT)\b': 'Telecom & Software',
    r'\b(SCREWFIX)\b': 'Housing & Utilities', 
    r'\b(EE D)\b': 'Telecom & Software',
    r'\b(NOTEMACHINE NOTEMACHINE)\b': 'Cash & Unknown'
}

df_tx['suggested_category_v2'] = None
df_tx['category_confidence_v2'] = 0.0
df_tx['match_type_v2'] = None

unmatched_clusters_v2 = []
unmatched_noise_indices_v2 = []

for cluster_id in df_tx['merchant_cluster_id'].unique():
    bool_idx = df_tx['merchant_cluster_id'] == cluster_id
    cluster_texts = df_tx.loc[bool_idx, 'cleaned_description'].tolist()
    
    if cluster_id == -1:
        for idx in df_tx[bool_idx].index:
            text = df_tx.loc[idx, 'cleaned_description']
            matched = False
            for pattern, category in regex_rules_v2.items():
                if re.search(pattern, text):
                    df_tx.at[idx, 'suggested_category_v2'] = category
                    df_tx.at[idx, 'category_confidence_v2'] = 1.0
                    df_tx.at[idx, 'match_type_v2'] = 'Rule'
                    matched = True
                    break
            if not matched:
                unmatched_noise_indices_v2.append(idx)
        continue
        
    cluster_resolved = False
    for pattern, category in regex_rules_v2.items():
        if any(re.search(pattern, text) for text in cluster_texts):
            df_tx.loc[bool_idx, 'suggested_category_v2'] = category
            df_tx.loc[bool_idx, 'category_confidence_v2'] = 1.0
            df_tx.loc[bool_idx, 'match_type_v2'] = 'Rule'
            cluster_resolved = True
            break
            
    if not cluster_resolved:
        unmatched_clusters_v2.append(cluster_id)
        
print(f"Phase 1: Rules Engine matched {len(df_tx[df_tx['match_type_v2'] == 'Rule'])} transactions.")

category_embeddings_v2 = model.encode(category_descriptions_v2)

if unmatched_noise_indices_v2:
    noise_embeddings = embeddings[unmatched_noise_indices_v2]
    sim_matrix = cosine_similarity(noise_embeddings, category_embeddings_v2)
    best_indices = np.argmax(sim_matrix, axis=1)
    best_scores = np.round(np.max(sim_matrix, axis=1), 2)
    
    df_tx.loc[unmatched_noise_indices_v2, 'suggested_category_v2'] = [category_labels_v2[i] for i in best_indices]
    df_tx.loc[unmatched_noise_indices_v2, 'category_confidence_v2'] = best_scores
    df_tx.loc[unmatched_noise_indices_v2, 'match_type_v2'] = 'ML'

for cluster_id in unmatched_clusters_v2:
    bool_idx = df_tx['merchant_cluster_id'] == cluster_id
    cluster_embeddings = embeddings[bool_idx]
    
    centroid = np.mean(cluster_embeddings, axis=0).reshape(1, -1)
    sim_matrix = cosine_similarity(centroid, category_embeddings_v2)
    best_idx = np.argmax(sim_matrix, axis=1)[0]
    best_score = np.round(np.max(sim_matrix, axis=1)[0], 2)
    
    df_tx.loc[bool_idx, 'suggested_category_v2'] = category_labels_v2[best_idx]
    df_tx.loc[bool_idx, 'category_confidence_v2'] = best_score
    df_tx.loc[bool_idx, 'match_type_v2'] = 'ML'
    
print(f"Phase 2: Semantic ML matched {len(df_tx[df_tx['match_type_v2'] == 'ML'])} transactions.")

display_cols = ['cleaned_description', 'merchant_cluster_id', 'match_type_v2', 'suggested_category_v2', 'category_confidence_v2']
display(df_tx[display_cols].sample(10))

Phase 1: Rules Engine matched 1 transactions.
Phase 2: Semantic ML matched 29 transactions.


,cleaned_description,merchant_cluster_id,match_type_v2,suggested_category_v2,category_confidence_v2
19,INTEREST ON YOUR STANDARD BALANC,0,ML,Savings & Investments,0.21
12,"PAYMENT, THANK YOU",1,ML,Insurance & Finance,0.25
5,"PAYMENT, THANK YOU",1,ML,Insurance & Finance,0.25
13,INTEREST ON YOUR STANDARD BALANC,0,ML,Savings & Investments,0.21
4,INTEREST ON YOUR STANDARD BALANC,0,ML,Savings & Investments,0.21
18,"PAYMENT, THANK YOU",1,ML,Insurance & Finance,0.25
25,"PAYMENT, THANK YOU",1,ML,Insurance & Finance,0.25
21,"MCDONALDS, STONEHOUSE",-1,Rule,Dining Out,1.00
23,"PAYMENT, THANK YOU",1,ML,Insurance & Finance,0.25
2,"PAYMENT, THANK YOU",1,ML,Insurance & Finance,0.25


In [12]:
output_cols = [
    'description', 
    'cleaned_description', 
    'merchant_cluster_id', 
    'suggested_category_v2', 
    'category_confidence_v2'
]
df_tx['category_confidence_v2'] = df_tx['category_confidence_v2'].round(3)
df_tx[output_cols].sort_values(['category_confidence_v2', 'suggested_category_v2']).to_csv('cleaned_transactions_preview_v2.csv', index=False)
print(f"Saved {len(df_tx)} categorized rows to cleaned_transactions_preview_v2.csv!")

Saved 30 categorized rows to cleaned_transactions_preview_v2.csv!
